# Little Bobby Tables

Building a query by pasting a name into a string turns data into code. SQLite
in memory, no setup.

In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE Students (name TEXT, grade REAL)")
db.executemany("INSERT INTO Students VALUES (?, ?)", [("Anna", 1.7), ("Ben", 2.3), ("Cem", 1.3)])

def lookup_unsafe(name):
    query = "SELECT * FROM Students WHERE name = '" + name + "'"      # concatenation
    print("   query:", query)
    return db.execute(query).fetchall()

print(lookup_unsafe("Anna"))

   query: SELECT * FROM Students WHERE name = 'Anna'
[('Anna', 1.7)]


## The injection

The input closes the quote and adds its own condition.

In [2]:
print(lookup_unsafe("' OR '1'='1"))       # every row, no DROP needed

   query: SELECT * FROM Students WHERE name = '' OR '1'='1'
[('Anna', 1.7), ('Ben', 2.3), ('Cem', 1.3)]


Bobby's full name needs two statements in one call. `sqlite3.execute` refuses
that (one of several protections a driver can offer); `executescript` does not.

In [3]:
bobby = "Robert'); DROP TABLE Students;--"
statement = "INSERT INTO Students (name) VALUES ('" + bobby + "')"
print(statement)
try:
    db.executescript(statement)
    db.execute("SELECT * FROM Students")
except sqlite3.OperationalError as e:
    print("after running it:", e)

INSERT INTO Students (name) VALUES ('Robert'); DROP TABLE Students;--')
after running it: no such table: Students


## The fix: keep data and code apart

In [4]:
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE Students (name TEXT)")
db.execute("INSERT INTO Students (name) VALUES (?)", (bobby,))          # parameterized
print(db.execute("SELECT name FROM Students").fetchall())
print(db.execute("SELECT * FROM Students WHERE name = ?", ("' OR '1'='1",)).fetchall())

[("Robert'); DROP TABLE Students;--",)]
[]


The value is sent separately from the query text and is never parsed as SQL,
whatever it contains.

**What we used:** SQL as an executable language, INSERT/DROP, and parameterized
queries (session 11, sections 8, 10 and 13).